In [1]:
import os

import branca.colormap as cm
import folium
import geopandas as gpd
import us
from branca.element import MacroElement
from gerrytools.data import geometries20
from jinja2 import Template

/Users/alexandramarcosquispe/Documents/mscapp/first-year/summer/San-Diego-Election-Analysis/.venv/lib/python3.13/site-packages/gerrytools/__init__.py:9: UserWarning: pygeos support was removed in 1.0. geopandas.use_pygeos is a no-op and will be removed in geopandas 1.1.
  geopandas.options.use_pygeos = False


Optional module 'mgrp' could not be imported: No module named 'docker'


In [2]:
# Council district boundaries, restricted to the City of San Diego's own
# districts (the source layer also covers other cities in the county).
# Their union defines the city boundary used to clip the demographic
# layer below.
city = gpd.read_file("../data/Municipal_Boundaries.geojson")
city = city.set_crs(epsg=4326) if city.crs is None else city.to_crs(epsg=4326)

city_boundary = city[city["Name"] == "NATIONAL CITY"]

city_boundary

DataSourceError: ../data/Municipal_Boundaries.geojson: No such file or directory

In [6]:
# Download (once) and load 2020 Census block-group geometries for California.
# These MGGG-processed shapefiles already carry P2 (race/ethnicity) table
# counts as attributes, so no separate Census API call is needed.
CA_BLOCKS = "../data/ca_districtr_block_view_v1.gpkg"
SAN_DIEGO_COUNTY_FIPS = "073"

if not os.path.exists(CA_BLOCKS):
    geometries20(us.states.CA, CA_BLOCKS, geometry="block")

blocks = gpd.read_file(CA_BLOCKS)
san_diego = blocks[blocks["COUNTYFP20"] == SAN_DIEGO_COUNTY_FIPS].copy()
san_diego = san_diego.to_crs(epsg=4326)

# Keep only block groups that actually fall within the City of San Diego
# (not the whole county, which also includes El Cajon, Chula Vista, etc.).
# Centroids are computed in a planar UTM CRS for accuracy, then tested
# against the city boundary.
utm_crs = san_diego.estimate_utm_crs()
centroids_utm = san_diego.geometry.to_crs(utm_crs).centroid
city_boundary_utm = gpd.GeoSeries([city_boundary], crs=san_diego.crs).to_crs(utm_crs).iloc[0]
san_diego = san_diego[centroids_utm.within(city_boundary_utm).to_numpy()].copy()

# Assign each block group to the council district whose polygon contains
# its centroid, so demographics can also be aggregated to the district
# level (see below).
centroids_utm = san_diego.geometry.to_crs(utm_crs).centroid
council_utm = council.to_crs(utm_crs)
bg_points = gpd.GeoDataFrame(
    {"GEOID20": san_diego["GEOID20"].values}, geometry=centroids_utm.values, crs=utm_crs
)
bg_district = gpd.sjoin(bg_points, council_utm[["DISTRICT", "geometry"]], predicate="within", how="left")
bg_district = bg_district[~bg_district.index.duplicated(keep="first")]
san_diego["DISTRICT"] = bg_district["DISTRICT"].values

# Simplify geometry so multiple choropleth layers stay a reasonable size in
# the browser/notebook (visually indistinguishable at city scale, ~0.0005
# deg is roughly 50m).
san_diego["geometry"] = san_diego["geometry"].simplify(0.0005, preserve_topology=True)

# Non-Hispanic-alone counts for White/Black/Asian, plus total Hispanic
# (of any race) - the standard PL 94-171 Table P2 breakdown.
RACE_COLUMNS = {
    "White": "WPOP20",
    "Black": "BPOP20",
    "Hispanic": "HISP20",
    "Asian": "ASIANPOP20",
}

# Also compute each group's share of the block group's total population,
# so the map can toggle between absolute counts and percentages.
safe_total = san_diego["TOTPOP20"].where(san_diego["TOTPOP20"] > 0, 1)
for label, col in RACE_COLUMNS.items():
    san_diego[f"{label}_pct"] = san_diego[col] / safe_total * 100

san_diego[["GEOID20", "DISTRICT", "TOTPOP20", *RACE_COLUMNS.values()]].head()

/Users/bsauvage/Documents/GitHub/mggg/San-Diego-Election-Analysis/.venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: Layer gerrydb_view_meta relies on the 'mggg_gerrydb' (JSON-formatted metadata for the view's tabular, geographic, and graph data.) extension that should be implemented in order to read it safely, but is not currently. Some data may be missing while reading that layer.
  return ogr_read(
/Users/bsauvage/Documents/GitHub/mggg/San-Diego-Election-Analysis/.venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: Layer gerrydb_geo_meta relies on the 'mggg_gerrydb' (JSON-formatted metadata for the view's geographies.) extension that should be implemented in order to read it safely, but is not currently. Some data may be missing while reading that layer.
  return ogr_read(
/Users/bsauvage/Documents/GitHub/mggg/San-Diego-Election-Analysis/.venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: Layer gerrydb_geo_attrs relies on the 

KeyError: 'COUNTYFP20'

In [4]:
# Aggregate block-group counts up to the council-district level, so the
# choropleth can toggle between census block groups and city council
# districts as the geographic unit.
district_totals = (
    san_diego.dropna(subset=["DISTRICT"])
    .groupby("DISTRICT")[["TOTPOP20", *RACE_COLUMNS.values()]]
    .sum()
    .reset_index()
)

council_demo = council.merge(district_totals, on="DISTRICT", how="left")
safe_total_district = council_demo["TOTPOP20"].where(council_demo["TOTPOP20"] > 0, 1)
for label, col in RACE_COLUMNS.items():
    council_demo[f"{label}_pct"] = council_demo[col] / safe_total_district * 100

council_demo[["DISTRICT", "NAME", "TOTPOP20", *RACE_COLUMNS.values()]]

,DISTRICT,NAME,TOTPOP20,WPOP20,BPOP20,HISP20,ASIANPOP20
0,2.0,Jennifer Campbell,147331,91605,4124,28717,11243
1,1.0,Joe LaCava,159578,98145,1990,18061,30174
2,8.0,Vivian Moreno,155775,16266,8488,115298,10978
3,6.0,Kent Lee,147678,51817,4714,21389,59472
4,3.0,Stephen Whitburn,151692,85863,8234,36350,10810
5,7.0,Raul Campillo,162198,82705,9144,36017,20918
6,9.0,Sean Elo-Rivera,157010,44167,15564,68603,20239
7,4.0,Henry Foster III,148545,14964,22872,70160,33022
8,5.0,Marni von Wilpert,155434,79063,3033,16810,45116


In [5]:
class RaceMetricControl(MacroElement):
    """A control panel with three independent radio-button groups - race,
    metric, and geographic unit - that together select one of several
    pre-built layers to show on the map, plus a legend that updates to
    match the current selection.
    """

    _template = Template(
        """
        {% macro html(this, kwargs) %}
        <div id="race-metric-{{ this.get_name() }}" style="
            position: fixed; top: 90px; right: 10px; z-index: 9999;
            background: white; padding: 10px 14px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.35); font-family: sans-serif;
            font-size: 12px; color: #222; line-height: 1.7;">
            <div style="font-weight:600;margin-bottom:4px;">Race / ethnicity</div>
            {% for race in this.races %}
            <label style="display:block;">
                <input type="radio" name="race-{{ this.get_name() }}" value="{{ race }}"
                       {% if race == this.default_race %}checked{% endif %}> {{ race }}
            </label>
            {% endfor %}
            <div style="font-weight:600;margin:8px 0 4px;">Metric</div>
            {% for key, meta in this.metrics.items() %}
            <label style="display:block;">
                <input type="radio" name="metric-{{ this.get_name() }}" value="{{ key }}"
                       {% if key == this.default_metric %}checked{% endif %}> {{ meta.label }}
            </label>
            {% endfor %}
            <div style="font-weight:600;margin:8px 0 4px;">Geographic unit</div>
            {% for key, meta in this.geographies.items() %}
            <label style="display:block;">
                <input type="radio" name="geo-{{ this.get_name() }}" value="{{ key }}"
                       {% if key == this.default_geo %}checked{% endif %}> {{ meta.label }}
            </label>
            {% endfor %}
        </div>

        <div id="demo-legend-{{ this.get_name() }}" style="
            position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 10px 14px; border-radius: 6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.35); font-family: sans-serif;
            font-size: 12px; color: #222;">
        </div>
        {% endmacro %}

        {% macro script(this, kwargs) %}
        (function() {
            var layerMap = {{ this.layer_map|tojson }};
            var legendData = {{ this.legend_data|tojson }};
            var map = {{ this._parent.get_name() }};
            var current = {
                race: {{ this.default_race|tojson }},
                metric: {{ this.default_metric|tojson }},
                geo: {{ this.default_geo|tojson }}
            };

            var legendEl = document.getElementById("demo-legend-{{ this.get_name() }}");

            function activeKey() { return current.race + "|" + current.metric + "|" + current.geo; }

            function renderLegend(key) {
                var d = legendData[key];
                if (!d) { return; }
                legendEl.innerHTML =
                    '<div style="font-weight:600;margin-bottom:4px;">' + d.label + '</div>' +
                    '<div style="width:180px;height:10px;border-radius:2px;background:' + d.gradient + ';"></div>' +
                    '<div style="display:flex;justify-content:space-between;font-size:11px;margin-top:2px;">' +
                    '<span>' + d.min + '</span><span>' + d.max + '</span></div>';
            }

            function setLayerVisible(key, visible) {
                var varName = layerMap[key];
                if (!varName || !window[varName]) { return; }
                if (visible) { map.addLayer(window[varName]); }
                else { map.removeLayer(window[varName]); }
            }

            function updateDimension(dim, value) {
                var oldKey = activeKey();
                current[dim] = value;
                var newKey = activeKey();
                if (newKey === oldKey) { return; }
                setLayerVisible(oldKey, false);
                setLayerVisible(newKey, true);
                renderLegend(newKey);
            }

            renderLegend(activeKey());

            document.querySelectorAll('input[name="race-{{ this.get_name() }}"]').forEach(function(input) {
                input.addEventListener('change', function() { updateDimension('race', this.value); });
            });
            document.querySelectorAll('input[name="metric-{{ this.get_name() }}"]').forEach(function(input) {
                input.addEventListener('change', function() { updateDimension('metric', this.value); });
            });
            document.querySelectorAll('input[name="geo-{{ this.get_name() }}"]').forEach(function(input) {
                input.addEventListener('change', function() { updateDimension('geo', this.value); });
            });
        })();
        {% endmacro %}
        """
    )

    def __init__(self, layer_map, legend_data, races, metrics, geographies, default_race, default_metric, default_geo):
        super().__init__()
        self._name = "RaceMetricControl"
        self.layer_map = layer_map
        self.legend_data = legend_data
        self.races = races
        self.metrics = metrics
        self.geographies = geographies
        self.default_race = default_race
        self.default_metric = default_metric
        self.default_geo = default_geo

In [6]:
# One perceptually-distinct sequential palette per demographic, scaled to
# that demographic's own min/max so each layer's spread is legible on its
# own terms (Black and White don't share a comparable range, whether in
# raw counts or percentages).
PALETTES = {
    "White": ["#f7fbff", "#08306b"],
    "Black": ["#fff5eb", "#7f2704"],
    "Hispanic": ["#f7fcf5", "#00441b"],
    "Asian": ["#fcfbfd", "#3f007d"],
}

METRICS = {
    "count": {"label": "Total population"},
    "pct": {"label": "% of population"},
}

GEOGRAPHIES = {
    "block_group": {"label": "Census block groups", "unit": "block group", "data": san_diego},
    "district": {"label": "City council districts", "unit": "council district", "data": council_demo},
}

DEFAULT_RACE, DEFAULT_METRIC, DEFAULT_GEO = "White", "count", "block_group"

centroid = san_diego.union_all().centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11, tiles="cartodbpositron")

layer_map = {}
legend_data = {}

for geo_key, geo_meta in GEOGRAPHIES.items():
    gdf = geo_meta["data"]
    is_district_geo = geo_key == "district"

    for label in RACE_COLUMNS:
        value_cols = {"count": RACE_COLUMNS[label], "pct": f"{label}_pct"}

        for metric_key, value_col in value_cols.items():
            key = f"{label}|{metric_key}|{geo_key}"
            vmax = float(gdf[value_col].max())
            colormap = cm.LinearColormap(PALETTES[label], vmin=0, vmax=vmax)
            metric_noun = "population" if metric_key == "count" else "% of pop."

            if is_district_geo:
                layer_data = gdf[["geometry", "DISTRICT", "NAME", value_col, "TOTPOP20"]]
                tooltip_fields = ["DISTRICT", "NAME", value_col, "TOTPOP20"]
                tooltip_aliases = ["District", "Council member", f"{label} {metric_noun}", "District total pop."]
                weight = 1.5
            else:
                layer_data = gdf[["geometry", value_col, "TOTPOP20"]]
                tooltip_fields = [value_col, "TOTPOP20"]
                tooltip_aliases = [f"{label} {metric_noun}", "Block group total pop."]
                weight = 0.3

            is_default = label == DEFAULT_RACE and metric_key == DEFAULT_METRIC and geo_key == DEFAULT_GEO

            # control=False keeps these layers out of the standard
            # LayerControl - they're driven entirely by RaceMetricControl below.
            fg = folium.FeatureGroup(name=key, show=is_default, control=False)
            folium.GeoJson(
                layer_data,
                style_function=lambda feature, colormap=colormap, value_col=value_col, weight=weight: {
                    "fillColor": colormap(feature["properties"][value_col]),
                    "color": "#666666",
                    "weight": weight,
                    "fillOpacity": 0.75,
                },
                tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True),
            ).add_to(fg)
            fg.add_to(m)
            layer_map[key] = fg.get_name()

            legend_data[key] = {
                "label": f"{label} — {metric_noun} per {geo_meta['unit']}",
                "gradient": f"linear-gradient(to right, {PALETTES[label][0]}, {PALETTES[label][1]})",
                "min": "0",
                "max": f"{vmax:,.0f}" if metric_key == "count" else f"{vmax:.1f}%",
            }

council_fg = folium.FeatureGroup(name="Council districts", show=True)
folium.GeoJson(
    council[["geometry", "JUR_NAME", "DISTRICT", "NAME"]],
    style_function=lambda feature: {"fillOpacity": 0, "color": "#222222", "weight": 1.5},
    tooltip=folium.GeoJsonTooltip(
        fields=["JUR_NAME", "DISTRICT", "NAME"],
        aliases=["City", "District", "Council member"],
    ),
).add_to(council_fg)
council_fg.add_to(m)

# Independent checkbox for the council-district boundary overlay (the
# demographic layers are excluded from this control, see control=False above).
folium.LayerControl(collapsed=False).add_to(m)

RaceMetricControl(
    layer_map=layer_map,
    legend_data=legend_data,
    races=list(RACE_COLUMNS.keys()),
    metrics=METRICS,
    geographies=GEOGRAPHIES,
    default_race=DEFAULT_RACE,
    default_metric=DEFAULT_METRIC,
    default_geo=DEFAULT_GEO,
).add_to(m)

m